# 15 · Model comparison (validation 2023, and 2020–2022 inner folds)

Every model with saved forecasts, on the same stocks and days. Intervals are 95% block-bootstrap
intervals over trading days (blocks of 20 days, 300 resamples), which respect the correlation between stocks on the
same day. Test-period results are produced once, by the freeze task 16.9.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.analysis.plots import (BLUE_CMAP, HORIZON_COLORS, MODEL_COLORS, TEXT_2, apply_style, bar_style,
                                dot_compare, figures_dir, save)
from src.config import Config
from src.evaluation.metrics import calibration_table, evaluate, join_labels, majority_guess, with_regimes
from src.models.data import label_frame, load_panel, regimes
from src.models.walkforward import run_or_load

cfg = Config.load()
apply_style()
pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)
val_start, val_end = cfg.period("validation")

In [ ]:
from src.evaluation.bootstrap import metrics_with_intervals
from src.models.walkforward import load_forecasts

panel = load_panel(cfg)
labels = label_frame(panel)
fc = load_forecasts(cfg, "validation")
joined = join_labels(fc, labels)
table = metrics_with_intervals(joined, metrics=("accuracy", "brier", "vol_mae"), n=300, block=20,
                               seed=cfg["models"]["seed"])
table.to_csv(figures_dir(cfg) / "validation_metrics.csv", index=False)
print("models:", ", ".join(sorted(fc.model.unique())))
table.round(4)

In [ ]:
direction_models = [m for m in fc.model.unique() if m != "hist_vol"]
vol_models = [m for m in fc.model.unique() if fc.loc[fc.model == m, "vol_forecast"].notna().any()]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dot_compare(axes[0], table, "accuracy", direction_models, panel.horizons, "accuracy_low", "accuracy_high",
            reference=0.5)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title("Direction accuracy (95% interval)")
dot_compare(axes[1], table, "brier", direction_models, panel.horizons, "brier_low", "brier_high", fmt="{:.3f}")
axes[1].set_title("Brier score (lower is better)")
fig.tight_layout()
save(fig, cfg, "15_direction_comparison")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
dot_compare(ax, table, "vol_mae", vol_models, panel.horizons, "vol_mae_low", "vol_mae_high", fmt="{:.4f}")
ax.set_title("Volatility forecast error, MAE (95% interval; lower is better)")
save(fig, cfg, "15_volatility_comparison")
plt.show()

## Calibration at 5 days

In [ ]:
cal = calibration_table(joined[joined.horizon_days == 5], bins=10)
fig, ax = plt.subplots(figsize=(5.6, 4.6))
for m in [m for m in ["tft", "lstm", "arima", "ma_crossover"] if m in cal.model.unique()]:
    g = cal[cal.model == m]
    ax.plot(g.p_mean, g.up_rate, marker="o", color=MODEL_COLORS[m], label=m)
lims = [0.3, 0.8]
ax.plot(lims, lims, color=TEXT_2, linewidth=1, linestyle="--")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("mean forecast p_up (decile)")
ax.set_ylabel("actual share of 5-day rises")
ax.set_title("Calibration, 5-day horizon")
ax.legend()
save(fig, cfg, "15_calibration_5d")
plt.show()

## Against the constant majority guess

The constant majority guess scores the share of days that went the majority way in the same period. It is a hindsight
reference line (EDA 02 §2), not a model. Balanced accuracy averages the hit rates on up and on down moves, so a
model that always says "up" scores 50% on it.

In [ ]:
ref = majority_guess(labels[labels.trade_date.isin(fc.trade_date.unique())])
print(ref.round(4).to_string(index=False))
acc = evaluate(fc[fc.model.isin(direction_models)], labels)
acc.pivot(index="model", columns="horizon_days", values=["accuracy", "balanced_accuracy"]).round(3)

## By market regime

Regimes from `src/features/regime.py`, as known before each day opens: India VIX terciles fitted on 2016–2022 and the
Nifty against its 200-day EMA. 2023 has almost no high-VIX or downtrend days, so these splits rest on few days; the
2020–2022 folds below cover both.

In [ ]:
lab_r = with_regimes(labels, regimes(cfg))
days = lab_r[lab_r.trade_date.isin(fc.trade_date.unique())].drop_duplicates("trade_date")
print("validation days by VIX band:", days.vix_regime.value_counts(normalize=True).round(3).to_dict())
print("validation days by trend:", days.trend_regime.value_counts(normalize=True).round(3).to_dict())
by_vix = evaluate(fc[fc.model.isin(direction_models)], lab_r, by=("model", "horizon_days", "vix_regime"))
by_vix.pivot_table(index=["horizon_days", "vix_regime"], columns="model", values="accuracy").round(3)

## Inner folds: walk-forward over 2020–2022

The same models, refitted on the same schedule, over three years inside the training period: the 2020 crash, the
2021 rally and the 2022 fall. Each fold trains only on rows whose labels end before it starts. The EDA that chose the
features also looked at these years, so this is a second check rather than a clean test.

In [ ]:
fc_in = load_forecasts(cfg, "inner")
joined_in = join_labels(fc_in, labels)
table_in = metrics_with_intervals(joined_in, metrics=("accuracy", "brier", "vol_mae"), n=300, block=20,
                                  seed=cfg["models"]["seed"])
table_in.to_csv(figures_dir(cfg) / "inner_metrics.csv", index=False)
ref_in = majority_guess(labels[labels.trade_date.isin(fc_in.trade_date.unique())])
print(ref_in.round(4).to_string(index=False))
table_in.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dot_compare(axes[0], table_in, "accuracy", direction_models, panel.horizons, "accuracy_low", "accuracy_high",
            reference=0.5)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title("Direction accuracy, 2020-2022 (95% interval)")
dot_compare(axes[1], table_in, "vol_mae", vol_models, panel.horizons, "vol_mae_low", "vol_mae_high", fmt="{:.4f}")
axes[1].set_title("Volatility MAE, 2020-2022 (lower is better)")
fig.tight_layout()
save(fig, cfg, "15_inner_comparison")
plt.show()

In [ ]:
by_year = evaluate(fc_in.assign(year=fc_in.trade_date.dt.year), labels, by=("model", "horizon_days", "year"))
ref_year = majority_guess(labels[labels.trade_date.isin(fc_in.trade_date.unique())].assign(
    year=lambda d: d.trade_date.dt.year), by=("horizon_days", "year"))
print(ref_year.pivot(index="horizon_days", columns="year", values="majority_accuracy").round(3).to_string())
by_year.pivot_table(index=["horizon_days", "model"], columns="year", values=["accuracy", "vol_mae"]).round(4)

In [ ]:
by_vix_in = evaluate(fc_in, lab_r, by=("model", "horizon_days", "vix_regime"))
by_vix_in.pivot_table(index=["horizon_days", "vix_regime"], columns="model", values=["accuracy", "vol_mae"]).round(4)

## Findings

1. **No model beats the constant base rate on plain accuracy, in 2023 or in 2020-2022.** Validation 2023: base rate 53.0% / 56.2% / 60.6% at 1 / 5 / 10 days; LSTM identical (it learned the base rate); ARIMA 47.8% / 55.4% / 58.8%; MA crossover 49.8% / 53.2% / 54.3%; TFT 51.1% / 51.1% / 54.4%. Inner folds 2020-2022: base rate 53.6% / 52.3% / 54.3%, the same as the hindsight majority guess (53.7% / 52.3% / 54.3%).
2. **Balanced accuracy:** only the TFT is above 50% in both periods, and barely (2023: 50.8% / 52.8% / 52.6%; 2020-2022: 50.4% / 50.8% / 51.4%). The MA crossover's small 2023 edge (50.8% / 51.0% / 50.5%) reverses in 2020-2022 (48.9% / 48.6% / 48.1%). ARIMA stays at 49-50% in both.
3. **The TFT's 2023 market timing does not hold up.** The rank correlation between its mean 5-day p_up and the mean 5-day return across days is +0.31 in 2023 but -0.04 over 2020-2022 (2020 -0.07, 2021 +0.06, 2022 +0.05); its within-day ranking of stocks moves from -0.06 to +0.02. The trained models carry no stable direction signal.
4. **Volatility is the trained models' useful output, and it holds:** the LSTM has the lowest error in both periods, in every year and in every VIX band. 2020-2022 MAE 0.0106 / 0.0074 / 0.0065 vs historical volatility's 0.0130 / 0.0084 / 0.0074 (18% / 12% / 12% lower); 2023 0.0069 / 0.0050 / 0.0045 vs 0.0088 / 0.0058 / 0.0050. On high-VIX days of 2020-2022 it is 0.0119 / 0.0084 / 0.0073 vs 0.0146 / 0.0095 / 0.0084. The TFT beats historical volatility in 2023 at 5 and 10 days but not in 2020-2022 (0.0142 / 0.0086 / 0.0081).
5. **2023 alone tests one regime:** India VIX was in its low band on 89% of validation days and never in its high band, and the Nifty was below its 200-day EMA on 9.4%. The 2020-2022 folds cover the crash and high-VIX months, which is where the MA crossover and the TFT's timing break down.
6. **What this sets for the project:** use the LSTM's volatility forecast for stops and position size, and don't treat any model's direction as an edge; direction has to come from event evidence (the agents). Before freezing the baselines (16.9): calibrate the probability levels on validation (15.2), choose the base forecast for the Forecaster agent on these results, then run the test period once with --freeze.